To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News

**Read our [blog post](https://unsloth.ai/blog/r1-reasoning) for guidance on how to train reasoning models.**

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
%%capture
!apt-get install swi-prolog

In [3]:
!pip install -U pyswip

In [4]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip install unsloth vllm
!pip install --upgrade pillow

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [5]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Load up `Llama 3.1 8B Instruct`, and set parameters

In [6]:
from unsloth import is_bfloat16_supported
import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

INFO 02-25 22:36:55 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-3b-instruct-bnb-4bit with actual GPU utilization = 69.63%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.15 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 368.
Unsloth: vLLM's KV Cache can use up to 52.7 GB. Also swap space = 6 GB.
INFO 02-25 22:37:08 config.py:549] This model supports multiple tasks: {'generate', 'embed', 'score', 'classify', 'reward

[W225 22:37:09.275901848 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 02-25 22:37:10 loader.py:1089] Loading weights with BitsAndBytes quantization.  May take a while ...
INFO 02-25 22:37:11 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-25 22:37:15 model_runner.py:1115] Loading model weights took 1.9356 GB
INFO 02-25 22:37:15 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-25 22:37:26 worker.py:267] Memory profiling takes 8.22 seconds
INFO 02-25 22:37:26 worker.py:267] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.70) = 55.12GiB
INFO 02-25 22:37:26 worker.py:267] model weights take 1.94GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 2.01GiB; the rest of the memory reserved for KV Cache is 51.08GiB.
INFO 02-25 22:37:27 executor_base.py:111] # cuda blocks: 92994, # CPU blocks: 10922
INFO 02-25 22:37:27 executor_base.py:116] Maximum concurrency for 2048 tokens per request: 726.52x
INFO 02-25 22:37:38 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory erro

Capturing CUDA graph shapes: 100%|██████████| 49/49 [01:36<00:00,  1.98s/it]

INFO 02-25 22:39:15 model_runner.py:1562] Graph capturing finished in 97 secs, took 7.14 GiB
INFO 02-25 22:39:15 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 117.27 seconds



Unsloth 2025.2.15 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [7]:
import re
from datasets import load_dataset, Dataset
import ast
import concurrent
import os
from pyswip import Prolog
import multiprocessing

os.environ["WANDB_PROJECT"] = "prolog-3b"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

# Load and prep dataset
SYSTEM_PROMPT = """
Generate a prolog solution for the asked question.
Follow these steps to craft your response:
1. reason about the given instruction
2. provide a high-quality prolog solution
3. write a query to verify the solution.
Output in the following format:
<reasoning>
...
</reasoning>
<code>
...
</code>
<query>
...
</query>

Write the query just inside <query></query> not in <code></code>.
"""

def extract_xml_knowledge(text: str) -> str:
    answer = text.split("<code>")[-1]
    answer = answer.split("</code>")[0]
    return answer.strip()

def extract_xml_query(text: str) -> str:
    answer = text.split("<query>")[-1]
    answer = answer.split("</query>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()


def split_prolog_rules(code):
    """
    Splits a string of multiple Prolog facts and rules by matching
    periods that are not part of a floating-point number. It removes
    any surrounding whitespace from each rule.
    """
    # This regex splits on a period that is NOT immediately followed by a digit.
    # \s* optionally matches any whitespace before and after the period.
    rules = re.split(r'\s*\.(?!\d)\s*', code)
    # Filter out any empty strings that may occur.
    return [rule for rule in rules if rule]

def parse_kb(prolog_code, query, answer):
    reward = 0
    try:
        prolog_interpreter = Prolog()
        # Filter out any empty rule
        #rules = [rule.strip() for rule in prolog_code.split(").") if rule.strip()]
        #rules = [rule + ")" for rule in rules]
        rules = split_prolog_rules(prolog_code)
        print("-"*20)
        print(rules)
        print("-"*20)
        for rule in rules:
            prolog_interpreter.assertz(rule)
            reward += 0.125
        # Use the instance's query method
        result = list(prolog_interpreter.query(query))
        for inference in result:
          for _, result_inference in inference.items():
            print("Expected: {}, Actual: {}".format(answer, result_inference))
            try:
              if float(result_inference) == float(answer):
                return 1+reward
            except:
              return reward
        #print(result)
        # Ensure that the comparison makes sense:
        # This assumes you expect a non-empty result when the answer is correct.
        return reward + 0.25
    except Exception as e:
        print(f"Error encountered: {e}")
        return reward

def run_in_separate_process(prolog_code, query, answer):
    # We'll use a multiprocessing.Queue to retrieve the result from the subprocess
    result_queue = multiprocessing.Queue()

    def worker():
        result = parse_kb(prolog_code, query, answer)
        result_queue.put(result)

    # Create the process
    process = multiprocessing.Process(target=worker)
    process.start()
    process.join()  # Wait for the process to finish

    # Retrieve and return the result (if available)
    if not result_queue.empty():
        return result_queue.get()
    return 0


def remove_prolog_comments_and_whitespace(code):
    # Remove block comments (/* ... */)
    code_no_block = re.sub(r'/\*[\s\S]*?\*/', '', code)
    # Remove single-line comments (% ...) from each line
    code_no_comments = re.sub(r'(?m)%.*$', '', code_no_block)
    # Remove newline and tab characters, but keep spaces
    cleaned_code = re.sub(r'[\n\t]', '', code_no_comments)
    return cleaned_code

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    reward = []
    for r, a in zip(responses, answer):
      knowledge_base = extract_xml_knowledge(r)
      knowledge_base = knowledge_base.replace("```prolog", "")
      knowledge_base = knowledge_base.replace("```", "")
      query = extract_xml_query(r)
      query = query.replace("```prolog", "")
      query = query.replace("```", "")
      query = query.replace("?-", "")
      query = query.strip()
      knowledge_base = remove_prolog_comments_and_whitespace(knowledge_base)
      print("#"*20)
      print(knowledge_base)
      print("#"*20)
      query = remove_prolog_comments_and_whitespace(query)
      reward_achieved = run_in_separate_process(knowledge_base, query, a)
      # check with ast if code can be parsed
      reward.append(reward_achieved)
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[-1]}", f"\nResponse:\n{responses[-1]}")
    return reward

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("<code>\n") == 1:
        count += 0.125
    if text.count("\n</code>\n") == 1:
        count += 0.125
    if text.count("\n<query>\n") == 1:
        count += 0.125
    if text.count("\n</query>") == 1:
        count += 0.125
        count -= (len(text.split("\n</query>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [8]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4, # Increase to 4 for smoother training
    num_generations = 8, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 1024,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 300,
    save_steps = 300,
    max_grad_norm = 0.1,
    report_to = "wandb", # Can use Weights & Biases
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 4
\        /    Total batch size = 32 | Total steps = 300
 "-____-"     Number of trainable parameters = 119,734,272
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: federico-p98 (halykoss) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


####################
:- dynamic price/2.price(tickets, 40).total_cost(NDiscountedTickets, TotalCost) :-    price(Tickets, Price),    NDiscountedTickets is 10,     TotalCost is NDiscountedTickets * Price.total_cost(12, TotalCost).
####################
--------------------
[':- dynamic price/2', 'price(tickets, 40)', 'total_cost(NDiscountedTickets, TotalCost) :-    price(Tickets, Price),    NDiscountedTickets is 10,     TotalCost is NDiscountedTickets * Price', 'total_cost(12, TotalCost)']
--------------------
Expected: 476, Actual: _2868
####################
ticket_price(40).tickets_bought(12).discount_tickets( tickets_bought - 10 ).full_price(10 * ticket_price).discounted_price( discount_tickets * (ticket_price * 0.95)).total_cost(full_price + discounted_price).?- total_cost(570).
####################
--------------------
['ticket_price(40)', 'tickets_bought(12)', 'discount_tickets( tickets_bought - 10 )', 'full_price(10 * ticket_price)', 'discounted_price( discount_tickets * (ticket_p

ERROR: Syntax error: Operator expected
ERROR: assertz((###
ERROR: ** here **
ERROR:  ReasoningTo solve this problem, we need to calculate the total cost of the concert tickets after applying the discount)). 


--------------------
[':- dynamic(ticket/3)', 'total_cost(Tickets) :-    ticket(Tickets, BaseCost, TotalCost)', 'ticket(Tickets, BaseCost, TotalCost) :-    N is Tickets - 10,    NewBaseCost is BaseCost * 0.95,    CostPerTicket is BaseCost - NewBaseCost,    TotalCost is CostPerTicket * N + BaseCost * 10', 'total_cost(12, Cost)']
--------------------
Expected: 476, Actual: _2866
####################
cost_of_ticket(40).total_cost(TotalCost) :-        N1 is 12,      N2 is 10,      DiscountedTickets is max(0, N1 - N2),      RemainingTickets is N1 - DiscountedTickets,          TotalDiscountedCost is DiscountedTickets * (cost_of_ticket(40) * 0.95),        TotalRemainingCost is RemainingTickets * cost_of_ticket(40),        TotalCost is TotalDiscountedCost + TotalRemainingCost.
####################
--------------------
['cost_of_ticket(40)', 'total_cost(TotalCost) :-        N1 is 12,      N2 is 10,      DiscountedTickets is max(0, N1 - N2),      RemainingTickets is N1 - DiscountedTickets,      

ERROR: assertz/1: No permission to modify static procedure `(is)/2'


...

--------------------
['sum_of_numbers(200, X, 2*X, 500)', 'findall(X, sum_of_numbers(200, X, 2*X, 500), Values)']
--------------------
Error encountered: Caused by: 'assertz((findall(X, sum_of_numbers(200, X, 2*X, 500), Values))).'. Returned: 'error(permission_error(modify, static_procedure, /(findall, 3)), context(:(system, /(assertz, 1)), _2998))'.
####################
sum_of_numbers(500).first_number(200).second_number(X, Y, Z) :- second_number_twice(X, Y, Z).third_number(X, Y, Z) :- third_number_value(X, Y, Z), sum_of_numbers(500).second_number_twice(X, Y, Z) :- Y is 2*Z.third_number_value(X, Y, Z) :- Z is (500 - X - Y) / 3.?- third_number(200, Y, Z).
####################


ERROR: assertz/1: No permission to modify static procedure `findall/3'
ERROR: Defined at /usr/lib/swi-prolog/boot/bags.pl:89


--------------------
['sum_of_numbers(500)', 'first_number(200)', 'second_number(X, Y, Z) :- second_number_twice(X, Y, Z)', 'third_number(X, Y, Z) :- third_number_value(X, Y, Z), sum_of_numbers(500)', 'second_number_twice(X, Y, Z) :- Y is 2*Z', 'third_number_value(X, Y, Z) :- Z is (500 - X - Y) / 3', '?- third_number(200, Y, Z)']
--------------------
Error encountered: Caused by: 'third_number(200, Y, Z).'. Returned: 'error(instantiation_error, context(:(system, /(is, 2)), _2954))'.
####################
x = 200.y is 2 * z.sum([x, y, z], 500).z is 500 - x - 2*y.
####################
--------------------
['x = 200', 'y is 2 * z', 'sum([x, y, z], 500)', 'z is 500 - x - 2*y']
--------------------
Error encountered: Caused by: 'assertz((x = 200)).'. Returned: 'error(permission_error(modify, static_procedure, /(=, 2)), context(:(system, /(assertz, 1)), _2944))'.
####################
find_third_number :-    write("Enter the value of the first number: "), read_first_number(X1),    write("Enter

ERROR: assertz/1: No permission to modify static procedure `(=)/2'


--------------------
['find_third_number :-    write("Enter the value of the first number: "), read_first_number(X1),    write("Enter the value of the third number: "), read_third_number(X3),    X2 is 2 * X3,    sum_of_numbers(X1, X2, X3), !', 'read_first_number(X) :-    write("The first number (200): "), read(X), X = 200', 'read_third_number(X) :-    write("The third number: "), read(X)', 'sum_of_numbers(X1, X2, X3) :-    Total is X1 + X2 + X3,    write("The sum of the numbers: "), writeln(Total), Total = 500']
--------------------


<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
